RAG pipeline.  Source text: PDFs from UK Financial Conduct Authority (FCA) online [Handbook](https://handbook.fca.org.uk/handbook), specifically the Conduct of Business Sourcebook (COBS) section, chapters 1-10A, last updated on 5 August 2026. 

First install requirements.

In [0]:
%pip install langchain langchain-community pypdf faiss-cpu sentence-transformers
dbutils.library.restartPython()

In [0]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

DATA_PATH = "fca_cobs_pdfs"
SAVE_PATH = "faiss_index" 

# load PDFs
loader = PyPDFDirectoryLoader(DATA_PATH)
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDFs.")

In [0]:
def clean_fca_text(text):
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        line = line.strip()
        # skip lines that are just R, G, N or COBS (case insensitive)
        if line.upper() in ['R', 'G', 'N', 'COBS', 'CHAPTER']:
            continue
        # skip lines that are just dots or whitespace
        if not line or re.match(r'^[\.\s\-_]+$', line):
            continue
        # skip lines that look like the handbook header/footer
        if 'www.handbook.fca.org.uk' in line or re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s\d{4}', line):
            continue
        cleaned_lines.append(line)
    # join back with single newlines
    return '\n'.join(cleaned_lines)

In [0]:
for doc in docs:
    doc.page_content = clean_fca_text(doc.page_content)

# chunk documents
# 1000 character chunk with 200 overlap to keep legal context intact
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks.")

import random
print(random.choice(chunks).page_content)

In [0]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# initialise embedding model 'all-MiniLM-L6-v2'
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# create FAISS vector store and save locally
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(SAVE_PATH)

print(f"Vector store successfully saved.")


In [0]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")


In [0]:
%pip install langchain-groq langchain
dbutils.library.restartPython()


In [0]:
import os
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# initialise components
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.load_local(SAVE_PATH, embeddings, allow_dangerous_deserialization=True)
llm = ChatGroq(model_name="groq/compound-mini", temperature=0)

# prompt template
prompt = ChatPromptTemplate.from_template("""
You are a professional UK Financial Compliance Assistant. 
Use the following retrieved context from the FCA COBS handbook to answer the question. 
If the answer is not in the context, say that you don't know. Do not hallucinate.

Context: {context}
Question: {question}

Helpful answer:""")

# build LCEL Chain
# format prompt -> pass to LLM -> parse output as string
rag_chain = (
    {"context": vector_store.as_retriever(search_kwargs={"k": 3}), "question": RunnablePassthrough()}
    | prompt 
    | llm 
    | StrOutputParser()
)

In [0]:
query = "What are the requirements for client categorisation?"
response = rag_chain.invoke(query)

print(response)